In [1]:
%load_ext autoreload
%autoreload 2

# Bar Eden

## Act 1.3
The Modular Transformer

The different axes that can be changed:
1. Positional encoding method:
    - RoPE:  Rotatory Positional Encoding with variable θ
    - ALiBI: Attention with linear biases
    - Sinusoidal: Fixed sinusoidal encoding
    - Learned: Learned positional encodings initialized to a scale ```scale```

2. Feed Forward network:
    - Gated vs non-gated
    - Non-linearity: GeLU, SiLU, Leaky ReLU

2. Attention:
    - head sharing: MHA (multi head attention),
                    GQA (grouped query attention),
                    MQA (multi query attention)
    - mask: fully causal (dense), sliding window, sparse block
    - score function: 
        - materialized: softmax, relu scores
        - kernelized: relu kernel, performer

3. Normalization:
    - Type: Layer Norm, RMS Norm, Scale Norm
    - Placement: Pre or Post

4. Residual:
    - Routing: Sequential or Parallel


Logging method:
- jsonl logging via analysis
- wandb logging: online and offline both

In [2]:
from src.datasets.dataset_wiki import make_wiki_dataset
from src.model_files.definitions import (
    ModelConfig,
    HyperParameters,
    AttentionConfig,
    FeedForwardConfig,
    PositionalConfig,
    BlockConfig,
    OptimizerConfig,
    TrainingConfig
)
from src.model_files.optimizer import AdamW
from src.modular_transformer import Transformer
from src.train import train_model   
from src.analysis import *

import torch
import os

### Config

In [3]:
# ⸻ Wandb ⸻
os.environ["WANDB_MODE"] = "offline"

In [4]:
# ⸻ Device ⸻

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.xpu.is_available():
    DEVICE = "xpu"
else:
    DEVICE = "cpu"

print(f"Using device: {DEVICE}")

Using device: xpu


In [5]:
# ── reproducibility ──
SEED = 42

#### Hyperparameters config

In [6]:
# ⸻ Hyperparameters ⸻
HYPERPARAMETERS = HyperParameters(
    device=DEVICE,
    seed=SEED,
    vocab_size=3000,
    context_size=256,
    embedding_dim=128,
    dropout_p=0.2
)

#### Per component configuration

In [7]:
# ⸻ Positional config ⸻
positional_cfg = PositionalConfig(
    kind="learned",
    scale=0.1  # required for learned positional configurations based on post_init
)

In [8]:
# ⸻ Attention config ⸻
attention_cfg = AttentionConfig(
    num_heads=4,
    num_groups=4,  # standard mha (num_groups == num_heads)
    dim_k=16,
    dim_v=32,
    mask="dense",
    score_function="softmax"
)

In [9]:
# ⸻ Feed forward netowrk config ⸻
ffn_cfg = FeedForwardConfig(
    kind="non_gated",
    activation="gelu",
    hidden_dim=512
)

In [10]:
# ⸻ Block config ⸻
block_cfg = BlockConfig(
    num_blocks=4,
    norm_placement="pre",
    residual="sequential"
)

#### Model Configuartion

In [11]:
# ⸻ Model config ⸻
model_config = ModelConfig(
    positional=positional_cfg,
    feedforward=ffn_cfg,
    attention=attention_cfg,
    block=block_cfg,
    hyperparameters=HYPERPARAMETERS
)

#### Training Configuration

In [12]:
# ⸻ Training config ⸻
train_config = TrainingConfig(
    log_path="runs.jsonl",
    lr_start=3e-3,
    lr_end=3e-4,
    iterations=500,
    log_interval=100,
    batch_size=128,
    max_norm=1.0,
    warmup_steps=500,
    wandb_project="Bar-eden-transformers"
)

#### Optimizer Configuration

In [13]:
# ⸻ Optimizer config ⸻
optimizer_config = OptimizerConfig(
    kind="adamw",
    betas=(0.9, 0.99),
    eps=1e-8,
    weight_decay=0.1,
    max_norm=1.0
)

### Dataset creation

In [14]:
# ⸻ Dataset ⸻
DATA_DIR = r"D:\Bar-Eden\Datasets\Wikitext-2"
data = make_wiki_dataset(
    data_dir=DATA_DIR, 
    device=DEVICE, 
    context_size=HYPERPARAMETERS.context_size, 
    vocab_size=HYPERPARAMETERS.vocab_size
)

trn  = data["train"]
dev  = data["dev"]
test = data["test"]

print(f"Vocab size: {HYPERPARAMETERS.vocab_size}")
print("Train input structure:", trn.inputs.shape)
print("Dev input structure:  ", dev.inputs.shape)

Vocab size: 3000
Train input structure: torch.Size([13106, 256])
Dev input structure:   torch.Size([1370, 256])


In [14]:
# ── diagnostic batch (shared across all models) ──
DIAG_X = trn.inputs[:64]
DIAG_Y = trn.targets[:64]

### Initializations

#### Transformer Initialization

In [15]:
# Transformer

Wikitext_transformer_model = Transformer(model_cfg=model_config, model_hp=HYPERPARAMETERS)

print("Optimized Modular Transformer Config:\n", Wikitext_transformer_model.config_dict())
print(f"Parameters: {sum(p.numel() for p in Wikitext_transformer_model.parameters()):,}")

Optimized Modular Transformer Config:
 {'positional': {'kind': 'learned', 'theta': None, 'scale': 0.1}, 'block': {'attention': {'num_heads': 4, 'num_groups': 4, 'dim_k': 16, 'dim_v': 32, 'mask': 'dense', 'window': None, 'block_size': None, 'num_local_blocks': 1, 'global_tokens': None, 'score_function': 'softmax'}, 'feedforward': {'kind': 'non_gated', 'activation': 'gelu', 'hidden_dim': 512, 'gelu_implementation': 'approximate', 'alpha': None}, 'normalization': {'kind': 'layer', 'bias': False, 'eps': 1e-05}, 'dropout': {'dropout_p': 0.2}, 'norm_placement': 'pre', 'residual': 'sequential'}, 'hp': {'device': 'xpu', 'seed': 42, 'vocab_size': 3000, 'context_size': 256, 'embedding_dim': 128, 'dropout_p': 0.2}}
Parameters: 1,141,376


#### Optimizer Initialization

In [16]:
optimizer = AdamW(Wikitext_transformer_model.parameters(), optimizer_config)

### Training

In [17]:
# Training run (Passing required optimizer and thermal pause duration)
wikitext_transformer_results = train_model(
    SEED=SEED, 
    model=Wikitext_transformer_model, 
    optimizer=optimizer, 
    train=trn, 
    dev=dev, 
    config=train_config, 
    run_name="Wikitext Modular Transformer", 
    device=DEVICE,
    pause_time=0  # minutes of thermal cooldown pause per checkpoint
)

  step     100 | dev 6.6983 vs train 6.6952 | lr 0.0006 | grad norm 0.139
  step     200 | dev 5.9769 vs train 6.0206 | lr 0.0012 | grad norm 0.204
  step     300 | dev 5.4934 vs train 5.5413 | lr 0.0018 | grad norm 0.212
  step     400 | dev 5.2088 vs train 5.2487 | lr 0.0024 | grad norm 0.300
  step     500 | dev 4.9909 vs train 5.0128 | lr 0.0030 | grad norm 0.273


dev_loss,█▅▃▂▁
grad_norm,▁▄▄█▇
lr,▁▃▄▆█
step,▁▃▅▆█
train_loss,█▅▃▂▁
ud_mean,▁▆▇██
clipped,False
dev_loss,4.99089
elapsed_active_seconds,740.7943
grad_norm,0.27339
lr,0.003


In [ ]:
# Saving artifacts
torch.save(wikitext_transformer_results, "Training results/wikitext_transformer_results.pt")
torch.save(Wikitext_transformer_model, "Model Instances/wikitext_transformer.pt")

In [ ]:
# Loading artifacts back
wikitext_transformer_model   = torch.load("Model Instances/wikitext_transformer.pt", weights_only=False)
wikitext_transformer_results = torch.load("Training results/wikitext_transformer_results.pt")

### Analysis of plots

In [ ]:
runs = load_runs("runs.jsonl")
plot_run_comparison(runs, metric="dev_loss")
plot_run_comparison(runs, metric="train_loss")
run_all_diagnostics(wikitext_transformer_model, wikitext_transformer_results, DIAG_X, DIAG_Y, "Wikitext Transformer")

### Sampling

In [ ]:
## Generating text
def generate(model, bpe, prompt, max_new_tokens, context_size, device,
             temperature=1.0, top_k=None):
    model.eval()
    encoded = bpe.encode(prompt)
    original_length = len(encoded)

    for _ in range(max_new_tokens):
        window = encoded[-context_size:]
        pad_len = context_size - len(window)
        padded = [0] * pad_len + window
        input_tokens = torch.tensor([padded], dtype=torch.long, device=device)

        with torch.no_grad():
            logits = model(input_tokens)[0, -1, :]   # (vocab,)

        logits = logits / temperature

        if top_k is not None:
            values, indices = torch.topk(logits, top_k)
            probs = torch.softmax(values, dim=-1)
            sampled = torch.multinomial(probs, num_samples=1)
            next_token = indices[sampled].item()
        else:
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

        encoded.append(next_token)

    return bpe.decode(encoded[original_length:])

In [ ]:
from src.bpe import ByteBPE

bpe = ByteBPE()
bpe.load(r"D:\Bar-Eden\Datasets\Wikitext-2\bpe.json")
print(generate(wikitext_transformer_model, bpe, "The history of", 400, HYPERPARAMETERS.context_size, DEVICE, temperature=0.8, top_k=40))